# Statcast Pitch Data — Exploratory Data Analysis

Pull one full season of Statcast data via `pybaseball` and explore the features relevant to pitch classification.

**Goals**
- Understand the shape and quality of the data
- Identify missing values and decide how to handle them
- Profile the distribution of pitch types (supervised learning labels)
- Visualize ball-flight features that will drive both models
- Surface correlations and redundancies across the feature set

## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pybaseball import statcast
from pybaseball import cache

cache.enable()  # cache Statcast downloads locally

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

SEED = 42
np.random.seed(SEED)

## 2. Load Data

Pull one full MLB season. 2023 is a good baseline — full 162-game schedule, universal DH, post-shift-ban rules in effect.

In [ ]:
# Full 2023 regular season
df_raw = statcast(start_dt='2023-03-30', end_dt='2023-10-01')
print(f"Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}")
df_raw.head(3)

## 3. Column Overview

In [ ]:
# All columns and dtypes
pd.DataFrame({'dtype': df_raw.dtypes, 'null_pct': df_raw.isnull().mean().mul(100).round(2)})

In [ ]:
# Candidate ball-flight features for modeling
BALL_FLIGHT_FEATURES = [
    'release_speed',       # velocity out of hand
    'effective_speed',     # perceived velocity at plate
    'release_spin_rate',   # spin rate (RPM)
    'spin_axis',           # spin axis (degrees)
    'pfx_x',               # horizontal break (inches, catcher's POV)
    'pfx_z',               # vertical break (induced, inches)
    'plate_x',             # horizontal location at plate
    'plate_z',             # vertical location at plate
    'release_pos_x',       # release point X
    'release_pos_y',       # release point Y (distance from mound)
    'release_pos_z',       # release point Z (height)
    'release_extension',   # extension toward plate
    'vx0', 'vy0', 'vz0',   # velocity components at release
    'ax', 'ay', 'az',      # acceleration components
]

print(f"{len(BALL_FLIGHT_FEATURES)} candidate features")
df_raw[BALL_FLIGHT_FEATURES].describe()

## 4. Missing Value Analysis

In [ ]:
missing = df_raw[['pitch_type'] + BALL_FLIGHT_FEATURES].isnull().mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
missing[missing > 0].plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('% Missing')
ax.set_title('Missing Values — Pitch Features')
plt.tight_layout()
plt.show()

print(missing[missing > 0].to_string())

## 5. Pitch Type Distribution

In [ ]:
# Statcast pitch type codes and their names
PITCH_TYPE_MAP = {
    'FF': 'Four-Seam Fastball',
    'SI': 'Sinker',
    'FC': 'Cutter',
    'SL': 'Slider',
    'CU': 'Curveball',
    'CH': 'Changeup',
    'FS': 'Split-Finger',
    'ST': 'Sweeper',
    'SV': 'Slurve',
    'KC': 'Knuckle Curve',
    'KN': 'Knuckleball',
    'EP': 'Eephus',
    'FO': 'Forkball',
    'SC': 'Screwball',
    'CS': 'Slow Curve',
}

counts = df_raw['pitch_type'].value_counts(dropna=False)
counts.index = counts.index.map(lambda x: f"{x} — {PITCH_TYPE_MAP.get(x, x)}" if pd.notna(x) else 'NaN')

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Pitch Count')
ax.set_title('Pitch Type Distribution — 2023 Season')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

pct = df_raw['pitch_type'].value_counts(normalize=True).mul(100).round(2)
pd.DataFrame({'count': df_raw['pitch_type'].value_counts(), 'pct': pct})

## 6. Filter to Pitches Used in Modeling

Drop rows with missing pitch type or missing core features, and restrict to pitch types with enough samples to model.

In [ ]:
MIN_SAMPLES = 1000  # drop pitch types too rare to model reliably

valid_types = df_raw['pitch_type'].value_counts()
valid_types = valid_types[valid_types >= MIN_SAMPLES].index.tolist()
print(f"Pitch types with >= {MIN_SAMPLES} samples: {valid_types}")

df = (
    df_raw
    .loc[df_raw['pitch_type'].isin(valid_types)]
    .dropna(subset=BALL_FLIGHT_FEATURES)
    .reset_index(drop=True)
)

print(f"\nFiltered dataset: {len(df):,} rows  ({len(df)/len(df_raw)*100:.1f}% of raw)")
print(f"Pitch types retained: {df['pitch_type'].nunique()}")

## 7. Feature Distributions by Pitch Type

Violin plots for the four most discriminating features.

In [ ]:
KEY_FEATURES = ['release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z']
FEATURE_LABELS = {
    'release_speed': 'Velocity (mph)',
    'release_spin_rate': 'Spin Rate (rpm)',
    'pfx_x': 'Horizontal Break (in)',
    'pfx_z': 'Vertical Break / Induced (in)',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

order = df['pitch_type'].value_counts().index.tolist()

for ax, feat in zip(axes, KEY_FEATURES):
    sns.violinplot(
        data=df, x='pitch_type', y=feat,
        order=order, ax=ax, inner='quartile', linewidth=0.8
    )
    ax.set_title(FEATURE_LABELS[feat])
    ax.set_xlabel('')

fig.suptitle('Feature Distributions by Pitch Type', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Movement Profile (Break Chart)

In [ ]:
# Sample for plotting to avoid overplotting
sample = df.groupby('pitch_type', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 2000), random_state=SEED)
)

fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette('tab10', n_colors=df['pitch_type'].nunique())

for i, pt in enumerate(order):
    sub = sample[sample['pitch_type'] == pt]
    ax.scatter(sub['pfx_x'], sub['pfx_z'], label=pt, alpha=0.25, s=8, color=palette[i])

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Horizontal Break (in)  ←Arm Side | Glove Side→')
ax.set_ylabel('Vertical Break / Induced (in)')
ax.set_title('Pitch Movement Profile — 2023 Season')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.01, 1), loc='upper left', markerscale=3)
plt.tight_layout()
plt.show()

## 9. Release Point by Pitch Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for i, pt in enumerate(order):
    sub = sample[sample['pitch_type'] == pt]
    ax.scatter(sub['release_pos_x'], sub['release_pos_z'], label=pt, alpha=0.2, s=8, color=palette[i])

ax.set_xlabel('Release Position X (ft)')
ax.set_ylabel('Release Position Z — Height (ft)')
ax.set_title('Release Point by Pitch Type')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.01, 1), loc='upper left', markerscale=3)
plt.tight_layout()
plt.show()

## 10. Correlation Matrix

In [ ]:
corr = df[BALL_FLIGHT_FEATURES].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 11. Class Balance Check

Important for supervised modeling — heavily imbalanced classes may need resampling or class weighting.

In [ ]:
class_counts = df['pitch_type'].value_counts()
ratio = class_counts.max() / class_counts.min()

fig, ax = plt.subplots(figsize=(9, 4))
class_counts.plot.bar(ax=ax, color='steelblue')
ax.set_xlabel('Pitch Type')
ax.set_ylabel('Count')
ax.set_title(f'Class Balance (max/min ratio: {ratio:.1f}x)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(class_counts.to_string())

## 12. Save Cleaned Dataset

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

cols_to_save = ['pitch_type', 'player_name', 'game_date', 'p_throws'] + BALL_FLIGHT_FEATURES
df[cols_to_save].to_parquet('../data/statcast_2023_cleaned.parquet', index=False)
print(f"Saved {len(df):,} rows to data/statcast_2023_cleaned.parquet")

## Summary

| Item | Finding |
|------|---------|
| Raw rows | TBD after run |
| Pitch types (>= 1k samples) | TBD |
| Key missing features | TBD |
| Most common pitch | TBD |
| Class imbalance ratio | TBD |
| Highly correlated features | TBD |

**Next steps**
- Decide on final feature set for supervised model (consider dropping highly correlated features)
- Assess whether class imbalance warrants SMOTE or class weights
- Consider handedness split (LHP vs RHP) — movement profiles mirror across arms